### Group info

ID: G5_4

Members:
- Vinicius Millan Santa Catharina
- Zaur Kambiev
- Stig Espen Paulsberg

### Imports

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from numpy.typing import ArrayLike, NDArray


### Task 1

#### File reader function

In [ ]:
def file_reader(file_path: Path | str) -> tuple[str, str]:
    '''Reads files and returns tuple of text content and encoding.
    
    Read through log files, first as UTF-8 then ISO-8859-1 if it fails.'''
    try:
        # Try reading as UTF-8
        with open(
            file=file_path,
            mode='r',
            encoding='utf-8'
        ) as file:
            file_text_content = file.read()
            encoding_used = file.encoding
    except UnicodeDecodeError:
        # If reading UTF-8 fails, read as ISO-8859-1 instead
        with open(
            file=file_path,
            mode='r',
            encoding='cp1252' # equivalent to iso-8859-1
        ) as file:
            file_text_content = file.read()
            encoding_used = file.encoding
    return file_text_content, encoding_used

#### Log parser function

In [ ]:
def log_parser(file_paths):
    '''Searches through file_paths looking for pattern matches.
    
    Returns a list of dictionaries representing valid transaction records.
    '''
    records = []
    
    # Patterns to match
    log_patterns = re.compile(
        r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s"
        r"\[(?P<register_id>[\w_]+)\]\s+"
        r"item=(?P<item>[A-Za-z]+)\s+"
        r"price=(?P<price>\d+\.\d+)\s+"
        r"(?P<currency>[A-Z]{3})\s+"
        r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$"
    )

    # Loop over every log_file
    for log_file in file_paths:
        # Loop over every line in text content of each log_file
        log_file_content = file_reader(log_file)[0].splitlines()
        for line in log_file_content:
            match = log_patterns.search(line.strip())
            if match:
                data = match.groupdict()
                record = {
                    "timestamp":    data["timestamp"],
                    "register_id":  data["register_id"],
                    "item":         data["item"],
                    "price":  float(data["price"]),
                    "currency":     data["currency"],
                    "status":       data["status"],
                }
                records.append(record)
    return records

In [8]:
data_log_files_path = Path('../data').resolve().rglob('*.log')

transaction_records = log_parser(data_log_files_path)

### Task 2

In [ ]:
from pathlib import Path

def directory_search(folder_name: str | Path) -> list[Path]:
    """Search a directory and its subdirectories for .log files.

    Args:
        folder_name: The path to the directory to search.

    Returns:
        A list of Path objects representing the matching log files.
    """
    files = []

    # Convert the input to a pathlib.Path object
    directory = Path(folder_name) 

    # Iterate over the items (file or directory) in the current directory
    for item in directory.iterdir():

        # Add matching log files to the results
        if item.is_file() and item.suffix == '.log':
            files.append(item)
            
        elif item.is_dir():
            # Recursively search the subdirectory
            files.extend(directory_search(item))

    return files               


folder = '../data'
print(directory_search(folder))

In [ ]:
import json

with open(r'../data/store_config.json', 'r') as file:
    content = json.load(file)


def inspect_json(content: dict | list) -> set[str]:
    """Recursively inspect JSON content and return active register ids.
    
        Args:
            content: A dictionary or list containing JSON data.
    
        Returns:
            A set of ids for registers whose active status is True.
        """
    ids = set()

    # Check whether the current content is a dictionary
    if type(content) is dict:
        for key, value in content.items():

            # Add the register id if the register is active
            if key == 'id' and content.get('active') is True:
                ids.add(content['id'])

            # Recursively inspect nested dictionaries and lists
            if type(value) is dict or type(value) is list:
                ids.update(inspect_json(value))

    # Check whether the current content is a list
    elif type(content) is list:
        for item in content:
            ids.update(inspect_json(item))
    
    return ids       


print(inspect_json(content))


### Task 3

Define a function for handling price outliers.

In [ ]:
def handle_outliers(prices: ArrayLike) -> NDArray[np.float64]:
    """Replace price outliers with the median of valid prices.

    Outliers are defined as prices more than three standard deviations
    away from the mean.

    Parameters
    ----------
    prices : array_like
        Prices convertible to a 1D NumPy array of floats.

    Returns
    -------
    ndarray
        A 1D NumPy array of prices with outliers replaced by the
        median of valid prices.
    """

    prices = np.asarray(prices, dtype=np.float64)

    if prices.ndim != 1:
        raise ValueError("Prices must be 1D.")

    mean = prices.mean()
    std = prices.std()

    valid_mask = abs(prices - mean) <= 3 * std

    median = np.median(prices[valid_mask])

    prices[~valid_mask] = median

    return prices

Define a function for calculating the refund ratio of items.

In [ ]:
EPS = 1e-6


def compute_refund_ratio(
        sales_count: NDArray,
        refund_count: NDArray
) -> NDArray:
    """Compute the refund-to-sale ratio for each item.

    A small positive constant (EPS) is added to the number of sales
    to prevent division by zero.

    Parameters
    ----------
    sales_count : ndarray
        Number of sales for each item.
    refund_count : ndarray
        Number of refunds for each item.

    Returns
    -------
    ndarray
        An array containing element-wise refund-to-sale ratios.
    """

    return refund_count / (sales_count + EPS)

### Task 4

Define a function that converts a list of transactions into a `pandas.DataFrame`.

In [ ]:
def list_to_df(transactions: list[dict]) -> pd.DataFrame:
    """Convert a list of transactions into a pandas DataFrame.

    Convert the timestamp column to datetime, set it as the DataFrame index,
    and sort the transactions chronologically.

    Parameters
    ----------
    transactions : list[dict]
        A list of dictionaries containing a `timestamp` field
        in YYYY-MM-DD HH:MM:SS format.

    Returns
    -------
    pd.DataFrame
        A DataFrame indexed by timestamp, sorted in chronological order.
    """

    transactions = pd.DataFrame(transactions)

    transactions["timestamp"] = pd.to_datetime(transactions["timestamp"])

    transactions = transactions.set_index("timestamp").sort_index()

    return transactions

Define a function that resamples transactions into 1-hour intervals. 

In [ ]:
def resample_hourly(transactions: pd.DataFrame) -> pd.DataFrame:
    """Aggregate transaction prices into hourly intervals by item.

    We group transactions only by item and ignore all differences
    in other columns (e.g. currency, status).

    Parameters
    ----------
    transactions : pd.DataFrame
        Transaction records with a DatetimeIndex and columns for item, price.

    Returns
    -------
    pd.DataFrame
        Hourly price totals grouped by item.
    """

    return (
        transactions
        .groupby("item")
        .resample("h")
        .agg(total_price=("price", "sum"))
    )

Define a function to correct price values for NOK transactions.

In [ ]:
def correct_nok_prices(transactions: pd.DataFrame) -> pd.DataFrame:
    """Correct the price glitch in NOK transactions."""
    
    result = transactions.copy()
    result.loc[result['currency'] == 'NOK', 'price']*=10

    return result

Define a function to add a new column containing prices converted to NOK.

In [ ]:
def add_nok_prices(transactions: pd.DataFrame) -> pd.DataFrame:
    """Add a column with transaction prices converted to NOK."""

    conversion_rates = {
        "NOK": 1.0,
        "EUR": 10.4,
        "USD": 9.3
    }

    result = transactions.copy()
    result['nok_price'] = (result['currency'].map(conversion_rates)*result['price']).round(2)

    return result

Define a function to create a pivot table displaying total transaction revenue grouped by item across status categories.

In [ ]:
def transaction_revenue_pivot(transactions: pd.DataFrame) -> pd.DataFrame:
    """Calculate total transaction revenue per item and status.

    Parameters
    ----------
    transactions : pd.DataFrame
        DataFrame containing transaction records with item, status,
        and nok_price columns.

    Returns
    -------
    pd.DataFrame
        Pivot table with items as rows, transaction statuses as columns,
        and summed NOK prices as values.
    """
    transactions_pivot = pd.pivot_table(
            data=transactions,
            values='nok_price',
            index='item',
            columns='status',
            aggfunc='sum'
    )

    return transactions_pivot

Define a function that calculates the refund ratio per item.

In [ ]:
def refund_ratio_per_item(transactions: pd.DataFrame) -> pd.Series:
    """Calculate the refund ratio for each item.

    Parameters
    ----------
    transactions : pd.DataFrame
        DataFrame containing transaction records with item and status columns.

    Returns
    -------
    pd.Series
        Refund ratio for each item, indexed by item.
    """

    transactions_pivot = pd.pivot_table(
                data=transactions,
                index='item',
                columns='status',
                aggfunc='size',
                fill_value=0
        )
    
    sales_count = transactions_pivot['SUCCESS'].to_numpy()
    refund_count = transactions_pivot['REFUND'].to_numpy()

    result = compute_refund_ratio(sales_count, refund_count)

    return pd.Series(result, index=transactions_pivot.index, name='refund_ratio')

### Task 5